In [ ]:
#@title 按這裡開始（先按 ▶）
print("✅ W14 出發！本週目標：自己寫切窗函式與特徵，用 KNN 分出四種活動")
print("資料一組錄一份共用，但切窗與特徵每個人都要自己在自己的電腦上寫一遍")
print("本週要自己補四個空：切片、往前跳、每段的平均、每段的標準差")

# W14　感測資料活動辨識（電腦教室版）

**手機只是資料來源，處理一律回到電腦上做。**
一組一支手機用 phyphox 錄，資料全組共用；但程式每個人都要自己寫一遍。

**錄資料的規格**
- phyphox 選 `Acceleration (without g)`，四種各錄 30 秒：
  `walk`、`run`、`still`、`stairs`。
- 四種都用**同一支手機、同一個口袋位置**，不然波形會亂掉。
- 錄完立刻 `Export Data → CSV (Comma, decimal point)`，
  存進雲端硬碟的 `AI115`，檔名就是 `walk.zip`、`run.zip`、`still.zip`、`stairs.zip`。

**開始之前**：功能表「檔案 → 在雲端硬碟中儲存副本」。

**還在等資料的人不要空等**：先跑第 1、2 格，或直接跑「備援資料」那一格先把流程走一遍。

### 第 1 格：解壓縮並看欄名

phyphox 匯出的是 zip，裡面才是 `Raw Data.csv`。不同手機、不同版本欄名可能不一樣，
所以先解壓縮再把欄名印出來看。

**這一格要做什麼**：掛載雲端硬碟、把四個 zip 解開。沒有空格，直接執行。

**寫對了會看到什麼**：印出一個 list，裡面有 `Time (s)` 與三個
`Linear Acceleration x/y/z (m/s^2)` 的完整名稱。

**找不到檔案**＝雲端硬碟還沒同步，回網頁版確認檔案在 `AI115` 裡再重跑一次。
**找不到 Raw Data.csv**＝版本檔名略有不同，跑 `!ls /content/walk` 看實際檔名。

In [ ]:
#@title 第 1 格：解壓縮並看欄名
from google.colab import drive
drive.mount("/content/drive")
import zipfile, glob, pandas as pd
d = "/content/drive/MyDrive/AI115/"
for k in ["walk", "run", "still", "stairs"]:
    zipfile.ZipFile(d + k + ".zip").extractall("/content/" + k)
f = glob.glob("/content/walk/*Raw*.csv")[0]
print(pd.read_csv(f).columns.tolist())

### 第 2 格：統一欄名並畫波形

只取前四欄（時間與 xyz），欄名一律改短成 `t`、`x`、`y`、`z` 好寫，
再多算一欄 `a`＝三軸合成的總加速度。

**這一格要做什麼**：讀四種活動、算總加速度、畫出 `walk` 的波形。沒有空格。

**寫對了會看到什麼**：印出四種活動各有幾筆（通常上千筆），
以及一張 `walk` 的波形圖。把最後一行的 `walk` 換成另外三種，**四張都要看過**。

**四張波形要看得出差別**：靜止是一條幾乎不動的線、走路是規律的小波浪、
跑步又高又密、上下樓介於中間但不規則。
四張長得一樣就回頭重錄，別急著訓練——那是拿法不同造成的。

In [ ]:
#@title 第 2 格：統一欄名並畫波形
import numpy as np
acts = {}
for k in ["walk", "run", "still", "stairs"]:
    f = glob.glob(f"/content/{k}/*Raw*.csv")[0]
    t = pd.read_csv(f).iloc[:, :4]
    t.columns = ["t", "x", "y", "z"]
    t["a"] = np.sqrt(t.x**2 + t.y**2 + t.z**2)
    acts[k] = t
print({k: len(v) for k, v in acts.items()})
acts["walk"]["a"].plot(figsize=(8, 2), title="walk")

### 備援：全組還沒錄好資料就跑這一格

**有自己的資料就跳過這一格。** 資料還沒上來的人先用課程附的示範資料把流程走完，
等一下把上面兩格重跑一次就會換成你們自己的資料。

示範資料只有 `still` 與 `walk` 兩類（前 60 秒靜止、後 60 秒走路），
所以正確率會比四類的時候高，**不能拿來當作業的數字**。

**寫對了會看到什麼**：印出 `備援資料：['still', 'walk']` 與兩類各幾筆。

In [ ]:
#@title 備援資料（投影片未含，執行所需）
import pandas as pd, numpy as np   # ←投影片未含，執行所需
RAW = "https://raw.githubusercontent.com/myliao2007/stust-course-1151/main/ai-intro-pc/data/"
t = pd.read_csv(RAW + "demo_motion.csv").iloc[:, :4]
t.columns = ["t", "x", "y", "z"]
t["a"] = np.sqrt(t.x**2 + t.y**2 + t.z**2)
acts = {"still": t[t.t < 60], "walk": t[t.t >= 60]}
print("備援資料：", list(acts), {k: len(v) for k, v in acts.items()})
print("這只是先讓流程跑得動，作業要用你們自己錄的四類資料")

### 第 3 格：自己寫切窗函式

感測資料是一長串數字，模型不能一次吃三千筆，要先切成一段一段。

- `win`：一段有幾筆。phyphox 的取樣率依手機而定，常見每秒 100 到 200 筆。
- `stride`：每次往前跳幾筆。`stride` 比 `win` 小就會重疊，段數變多，邊界也不容易漏掉。

**這一格要做什麼**：補兩行。
- 第一行：取出 `arr` 的第 `i` 到 `i+win` 筆（就是切片，結束位置是 `i+win` 不是 `win`）。
- 第二行：往前跳 `stride` 筆（寫成跳 `win` 就完全不重疊，段數會少一半）。

**寫對了會看到什麼**：印出「切出 XX 段，每段 100 筆」，**段數一定要大於 0**。

切出 0 段幾乎都是 `win` 設得比整段資料還長，把 `win` 調小再試。

In [ ]:
#@title 第 3 格：自己寫 windows()
def windows(arr, win, stride):
    """把一長串數字切成一段一段，回傳一個 list"""
    out = []
    i = 0
    while i + win <= len(arr):
        out.append(____)   # ← 自己寫：arr 的第 i 到 i+win 筆
        i = ____           # ← 自己寫：往前跳 stride 筆
    return out
segs = windows(acts["walk"]["a"].values, 100, 50)
print("切出", len(segs), "段，每段", len(segs[0]), "筆")

### 第 4 格：自己寫特徵並訓練

為什麼特徵挑「平均」與「標準差」就夠了：
平均代表用力程度，標準差代表晃動劇不劇烈——這兩個數字就足以分開靜止、走路與跑步。

**這一格要做什麼**：補兩個空，算出每一段的平均與標準差。
`seg` 是一個 numpy 陣列，直接用它的方法即可。

**寫對了會看到什麼**：印出一個 0 到 1 的正確率，以及一個方陣（混淆矩陣）。
四類資料通常落在 0.8 以上；用備援的兩類資料會更高，那不算數。

**正確率剛好 1.0 要懷疑**：檢查是不是四個檔其實是同一份
（忘了匯出就接著錄下一種）。

In [ ]:
#@title 第 4 格：自己寫 feats() 並訓練
def feats(t, label, win=100, stride=50):
    rows = []
    for seg in windows(t["a"].values, win, stride):
        rows.append({"m": ____,      # ← 自己寫：這段的平均
                     "s": ____,      # ← 自己寫：這段的標準差
                     "y": label})
    return pd.DataFrame(rows)
data = pd.concat([feats(t, k) for k, t in acts.items()])
X, y = data[["m", "s"]], data["y"]
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.3, random_state=0)
knn = KNeighborsClassifier(n_neighbors=5).fit(Xtr, ytr)
p = knn.predict(Xte)
print("正確率 =", round(accuracy_score(yte, p), 3))
print(confusion_matrix(yte, p))

### 第 5 格（進階）：掃描三種窗長

三種 `win` 各跑一次，把段數與正確率抄下來，作業要交。沒有空格，直接執行。

**寫對了會看到什麼**：三行 `win = ... 段數 = ... 正確率 = ...`。

**要回答的問題**：段數少但每段長，或段數多但每段短，哪個好？用數字回答，
不要用「感覺」回答。

In [ ]:
#@title 第 5 格（進階）：掃描窗長
from sklearn.model_selection import cross_val_score
for w in [50, 100, 200]:
    d2 = pd.concat([feats(t, k, win=w, stride=w // 2)
                    for k, t in acts.items()])
    sc = cross_val_score(KNeighborsClassifier(5),
                         d2[["m", "s"]], d2["y"], cv=3)
    print("win =", w, "段數 =", len(d2),
          "正確率 =", round(sc.mean(), 3))

### 收工：延伸挑戰與繳交

- **A**（每個人都要做完）：`win` 依序改 50、100、200，`stride` 都設成 `win` 的一半，
  三個正確率都記下來——這三個數字就是作業內容。
- **B**：用 phyphox 的 `Gyroscope` 再錄一次，多算兩個特徵接進去，看正確率升或降。
- **C**：說出走路與上下樓為什麼最容易被搞混，並從你的波形圖指出一段當證據。

**常見狀況**：讀進來只有一欄＝匯出選成分號分隔，重匯出選 `CSV (Comma, decimal point)`；
欄位不是四欄＝實驗選錯，要選 `Acceleration (without g)`；
切出 0 段＝`win` 比資料還長。

In [ ]:
#@title 收工檢查（直接按 ▶）
print("本週要交：四種活動的波形圖（四張）、寫完 windows() 與 feats() 的 .ipynb")
print("以及三種 win 的正確率與混淆矩陣")
print("檔名：AI導論_W14_學號_姓名，上傳課程表單，下次上課前一天 23:59")
print("提醒：資料與筆記本一律存雲端硬碟，教室電腦重開機會還原")

---

<details>
<summary>參考解（四個空格都自己試過再打開）</summary>

```python
# 第 3 格
        out.append(arr[i:i + win])
        i = i + stride

# 第 4 格
        rows.append({"m": seg.mean(),
                     "s": seg.std(),
                     "y": label})
```

為什麼是這樣寫：

- `arr[i:i+win]` 是切片：從第 `i` 筆開始、取 `win` 筆。
  寫成 `arr[i:win]` 只有第一段是對的，之後會越切越短。
- `i = i + stride` 才會重疊。寫成 `i = i + win` 就完全不重疊，段數少一半，
  剛好跨在兩段之間的動作也會被切壞。
- `seg` 是 numpy 陣列，`seg.mean()`、`seg.std()` 直接算。
  （numpy 的 `std()` 預設是母體標準差，pandas 預設是樣本標準差，
  兩者差一點點，本週不影響結論。）

</details>